In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!pip install scipy PyWavelets -q

import requests, io
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import scipy.io as iomat
import helpers
from TIMBRE import TIMBRE
import pywt

# --- Load data ---
url = "https://api.figshare.com/v2/articles/24757638"
files = requests.get(url).json()['files']
f = next(x for x in files if x['name'] == 'data04.mat')
data = iomat.loadmat(io.BytesIO(requests.get(f['download_url']).content))

# --- Preprocess ---
LFPs = helpers.filter_data(data['lfps'], 2, fs=25, use_hilbert=True)
test_inds, train_inds = helpers.test_train(data['lapID'], which_phase=2, n_folds=5, which_fold=0)
wLFPs, U, Xv = helpers.whiten(LFPs, train_inds)
print(f"LFPs shape: {LFPs.shape}, electrodes: {LFPs.shape[1]}")

In [54]:
wLFPs

array([[ 0.73999841-0.62574027j,  0.38053384+0.63292897j,
         0.87532264-0.30498332j, ..., -0.60551548-0.2177046j ,
         1.00169958+0.13309589j,  0.36298052-0.20244512j],
       [ 1.50053524+1.18364299j, -1.07589831+0.65793898j,
         0.41575478+0.9627985j , ...,  0.1495114 -1.20343489j,
         0.04547853+1.71450082j,  0.4000686 +0.59543352j],
       [-1.02352849+1.61195908j, -0.5539119 -0.97416232j,
        -0.38253115-0.21877552j, ...,  1.15463521-0.23526446j,
        -1.20317961+0.40540573j, -0.44624161+0.35448579j],
       ...,
       [-0.76123334-0.11893911j,  0.19479765-0.34267333j,
         1.16992937-0.44220222j, ..., -0.07185263-0.08446722j,
        -0.02203455+0.01992545j, -0.01015581+0.02942245j],
       [ 0.24524846-0.55551893j,  0.19141178+0.22520785j,
         0.52506678+0.25787688j, ..., -0.09572702-0.13566928j,
         0.28699559+0.05715382j,  0.09359373-0.07573245j],
       [ 0.20694334-0.05423474j,  0.0312122 -0.01425613j,
         0.04970857-0.10217451

In [2]:
m3,_,_= TIMBRE(wLFPs, data['lapID'][:, 1], test_inds, train_inds, hidden_nodes=3)
m6,_,_= TIMBRE(wLFPs, data['lapID'][:, 1], test_inds, train_inds, hidden_nodes=6)

t3 = get_complex_weights(m3,3)
t6 = get_complex_weights(m6,6)

In [13]:
raw_complex = helpers.layer_output(wLFPs[stay_inds], m3, 0)
phase = np.angle(raw_complex[:, :3] + 1j * raw_complex[:, 3:])


1111/1111 ━━━━━━━━━━━━━━━━━━━━ 0s 214us/step


In [15]:
stay_inds = np.where(data['lapID'][:, 3] == 2)[0]

In [35]:
count = 0

for i in range(len(stay_inds) - 1):
    if stay_inds[i+1] - stay_inds[i] > 1:
        count += 1 
        #print('Jump is from ' + str(i) + ' to ' + str(i+1))
print(count)

98


In [37]:
# find where stay_inds transitions from False to True — each transition is a new bout
stay_bool = np.zeros(len(wLFPs), dtype=bool)
stay_bool[stay_inds] = True

bout_starts = np.where(np.diff(stay_bool.astype(int)) == 1)[0] + 1
bout_ends   = np.where(np.diff(stay_bool.astype(int)) == -1)[0] + 1

bout_lengths = bout_ends - bout_starts
print(f"Number of stay bouts: {len(bout_starts)}")
print(f"Bout lengths — min: {bout_lengths.min()}, max: {bout_lengths.max()}, mean: {bout_lengths.mean():.1f}")

Number of stay bouts: 99
Bout lengths — min: 267, max: 1158, mean: 359.1


In [56]:
from scipy.stats import pearsonr, ttest_1samp
from scipy.ndimage import uniform_filter1d

# ── 1. THETA CARRIER AND PHASE ────────────────────────────────────────────────
# SVD of the raw LFP on training data
# v[0, :] is the first right singular vector — the direction of maximum variance
# in electrode space, which corresponds to the global theta oscillation (PC1)
_, _, v = np.linalg.svd(LFPs[train_inds, :], full_matrices=False)

# project all timepoints onto this direction to get a single theta timeseries
# np.conj because LFPs is complex-valued (after Hilbert transform)
theta_carrier = LFPs @ np.conj(v[0, :])   # (T,) complex

# normalize to unit amplitude so we only retain phase information
# then extract instantaneous phase at each timepoint, range [-pi, pi]
theta_phase = np.angle(theta_carrier / np.abs(theta_carrier))   # (T,)


# ── 2. NODE PHASE FROM ComplexDense LAYER ────────────────────────────────────
# TIMBRE internally splits complex LFP into real and imaginary halves
# and concatenates them: [Re(wLFPs) | Im(wLFPs)] → shape (T, 384)
# we replicate this here so our manual matrix multiply matches what the layer did
X_input = np.concatenate([np.real(wLFPs), np.imag(wLFPs)], axis=1)   # (T, 384)

# extract the two weight matrices from the ComplexDense layer
# Wr: real part of weights (192, 3)
# Wi: imaginary part of weights (192, 3)
Wr, Wi = m3.layers[0].get_weights()

# manually compute the complex-valued output of ComplexDense before abs()
# this is standard complex multiplication: (a + bi)(c + di) = (ac-bd) + (ad+bc)i
# where a = Re(wLFP), b = Im(wLFP), c = Wr, d = Wi
raw_real = X_input[:, :192] @ Wr - X_input[:, 192:] @ Wi   # (T, 3) real part
raw_imag = X_input[:, :192] @ Wi + X_input[:, 192:] @ Wr   # (T, 3) imag part

# combine into complex and extract phase — this is the phase of each node's
# complex activation before TIMBRE discards it with abs()
node_phase = np.angle(raw_real + 1j * raw_imag)   # (T, 3)


# ── 3. SMOOTH PHASE HELPER ───────────────────────────────────────────────────
def smooth_phase(phase, window=25):
    """
    Smooths a phase timeseries over a sliding window while correctly
    handling circular wraparound at ±pi.
    
    Naive averaging of phase values fails because the average of +pi and -pi
    should be ±pi (same point on circle) not 0. To fix this:
    1. convert phase to a point on the unit circle (complex exponential)
    2. average real and imaginary parts separately (standard linear averaging)
    3. convert back to phase angle
    
    window=25 = 1 second at 25 Hz — removes fast theta oscillations,
    reveals only slow drift
    """
    # convert phase angle to point on unit circle
    z = np.exp(1j * phase)
    # average real and imaginary components separately over the window
    z_smooth = uniform_filter1d(z.real, window) + 1j * uniform_filter1d(z.imag, window)
    # convert back to phase angle
    return np.angle(z_smooth)


# ── 4. COMPUTE CORRELATION PER BOUT PER NODE ─────────────────────────────────
# for each stay bout, correlate each node's smoothed relative phase
# with the smoothed theta amplitude
# relative phase = node phase minus theta carrier phase
# if negative correlation: when theta is strong, node phase is more negative
# (phase-locked to carrier); when theta weakens, phase drifts positive

corr_array = np.zeros((len(bout_starts), 3))   # (n_bouts, n_nodes)

for i in range(len(bout_starts)):
    # extract this bout's timepoints
    s, e = bout_starts[i], bout_ends[i]

    # compute relative phase: node phase minus theta carrier phase
    rp = node_phase[s:e, :] - theta_phase[s:e, None]   # (bout_len, 3)

    # wrap to [-pi, pi] using exp/angle trick to handle circular boundary
    rp = np.angle(np.exp(1j * rp))

    # smooth each node's relative phase over 1-second window
    rp_smooth = np.zeros_like(rp)
    for node in range(3):
        rp_smooth[:, node] = smooth_phase(rp[:, node], window=25)

    # smooth theta amplitude over same 1-second window
    # np.abs() of complex theta_carrier gives instantaneous amplitude
    theta_amp_smooth = uniform_filter1d(np.abs(theta_carrier[s:e]), 25)

    # pearson correlation between each node's smoothed relative phase
    # and smoothed theta amplitude for this bout
    for node in range(3):
        r, _ = pearsonr(rp_smooth[:, node], theta_amp_smooth)
        corr_array[i, node] = r


# ── 5. T-TEST AGAINST ZERO ACROSS ALL BOUTS ──────────────────────────────────
# tests whether the mean correlation across 99 bouts is significantly
# different from zero — i.e. is the phase-amplitude relationship consistent
# or just random bout-to-bout variation
print("Mean phase-amplitude correlation per node:")
for node in range(3):
    mean_r = corr_array[:, node].mean()
    std_r  = corr_array[:, node].std()
    _, p   = ttest_1samp(corr_array[:, node], 0)
    print(f"Node {node}: mean r = {mean_r:.3f} ± {std_r:.3f}, p = {p:.4f}")

Mean phase-amplitude correlation per node:
Node 0: mean r = -0.150 ± 0.167, p = 0.0000
Node 1: mean r = 0.012 ± 0.197, p = 0.5444
Node 2: mean r = -0.094 ± 0.224, p = 0.0001
